
# ZynNova：极复杂全电池微结构 → 通用多域自适应非结构化 TetGen → COMSOL MPHTXT

这个版本专门验证这次源码升级：

1. **正极 30 个颗粒全部属于一个 FEM/COMSOL 材料域** `positive_active`；
2. **负极 30 个颗粒全部属于一个 FEM/COMSOL 材料域** `negative_active`；
3. 颗粒仍保留独立 tracking ID，便于统计每个颗粒，但 tracking ID 在 PLC 提取**之前**通过 `material_region_map` 合并，因此接触颗粒之间不会产生虚假的材料界面；
4. 正极 / 隔膜 / 负极三个宏观区域严格保持长方体；
5. 使用新的 `TetGenMeshingConfig.fixed_interface_pairs` 锁定两个隔膜平面；
6. 使用新的统一入口 `mesh_unstructured_regions(...)`，任何复杂标签体素都走：
   `labels -> conforming PLC -> TetGen C++ -> nonuniform Tet4`；
7. COMSOL 导出兼容旧 notebook 参数 `include_boundary_triangles=True`，不会再出现 `TypeError`；
8. 最终验证 COMSOL 中 `positive_active` 和 `negative_active` 各自都只有一个 3D domain entity。

> 运行前：
>
> ```powershell
> python -m pip install -e ".[zynmorph-tetgen]" -v
> python scripts\diagnose_tetgen_native.py
> ```


In [ ]:

from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import generate_binary_structure, label as ndi_label

try:
    import zynnova
except ModuleNotFoundError:
    candidates = [Path.cwd(), *Path.cwd().parents]
    explicit_root = os.environ.get("ZYNNOVA_PROJECT_ROOT")
    if explicit_root:
        candidates = [Path(explicit_root), *candidates]
    project_root = next(
        (p for p in candidates if (p / "src" / "zynnova").is_dir()),
        None,
    )
    if project_root is None:
        raise
    sys.path.insert(0, str(project_root / "src"))
    import zynnova

from zynnova.zynmorph import (
    BatteryPhase,
    IrregularMeshPolicy,
    LocalRefinementZone,
    MicrostructureVolume,
    TetGenMeshingConfig,
    audit_multiphase_plc,
    count_nonmanifold_voxel_edges,
    export_fem_mesh,
    extract_multiphase_plc,
    inspect_comsol_mphtxt,
    load_comsol_tet4_mphtxt,
    lock_plc_interfaces,
    mesh_complex_regions,
    mesh_unstructured_regions,
    regularize_nonmanifold_junctions,
    smooth_multiphase_plc,
    tetgen_native_diagnostics,
    tetgen_native_status,
)
from zynnova.geometry import (
    tetrahedron_mean_ratio,
    tetrahedron_signed_volumes,
)

print("ZynNova:", zynnova.__file__)
print("Python :", sys.executable)


## 1. 全电池尺寸与复杂度

In [ ]:

SEED = 20260818

# 数组顺序 (z, y, x)，厚度方向为 x。
NZ = 64
NY = 64
CATHODE_NX = 32
SEPARATOR_NX = 16
ANODE_NX = 32
NX = CATHODE_NX + SEPARATOR_NX + ANODE_NX

VOXEL_SIZE_M = 0.80e-6

N_CATHODE_PARTICLES = 30
N_ANODE_PARTICLES = 30
TARGET_ACTIVE_FRACTION = 0.70
ELECTROLYTE_SKIN_VOXELS = 1

RUN_TETGEN = os.environ.get("ZYNNOVA_RUN_NATIVE_TETGEN", "1") == "1"
MAXIMUM_TETRAHEDRA = 8_000_000

OUTPUT_DIR = Path("zynnova_runs") / "complex_full_cell_unstructured_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

P = BatteryPhase
PHASE_NAMES = {
    int(P.SEPARATOR_ELECTROLYTE): "separator_electrolyte",
    int(P.POSITIVE_ACTIVE): "positive_active",
    int(P.POSITIVE_ELECTROLYTE): "positive_electrolyte",
    int(P.NEGATIVE_ACTIVE): "negative_active",
    int(P.NEGATIVE_ELECTROLYTE): "negative_electrolyte",
}

print("Voxel shape (z,y,x):", (NZ, NY, NX))
print(
    "Physical size [um] (z,y,x):",
    (np.asarray((NZ, NY, NX)) * VOXEL_SIZE_M * 1e6).tolist(),
)
print(
    "Layer thickness [um]:",
    {
        "cathode": CATHODE_NX * VOXEL_SIZE_M * 1e6,
        "separator": SEPARATOR_NX * VOXEL_SIZE_M * 1e6,
        "anode": ANODE_NX * VOXEL_SIZE_M * 1e6,
    },
)



## 2. 生成每极 30 个形状各异的颗粒

颗粒不是简单球体。每个颗粒独立随机：

- 三轴半径；
- 3D 旋转；
- superquadric 指数；
- 多频表面粗糙度；
- 片状 / 棒状 / 三轴 / 紧凑等形状族。

`2 × 5 × 3 = 30` 的空间分层种子保证每个颗粒都真正出现；全局距离排序再把活性材料体素数离散校准到 70%。


In [ ]:

def random_rotation_matrix(rng: np.random.Generator) -> np.ndarray:
    matrix = rng.normal(size=(3, 3))
    q, _ = np.linalg.qr(matrix)
    if np.linalg.det(q) < 0.0:
        q[:, 0] *= -1.0
    return q


def build_particle_specs(
    shape_zyx: tuple[int, int, int],
    *,
    rng: np.random.Generator,
    n_particles: int,
    electrode_name: str,
):
    if n_particles != 30:
        raise ValueError("This notebook intentionally uses exactly 30 particles per electrode.")

    nz, ny, nx = shape_zyx
    specs = []
    pid = 0

    for iz in range(2):
        for iy in range(5):
            for ix in range(3):
                center = np.array(
                    [
                        (iz + 0.5) * nz / 2 + rng.uniform(-0.13, 0.13) * nz / 2,
                        (iy + 0.5) * ny / 5 + rng.uniform(-0.15, 0.15) * ny / 5,
                        (ix + 0.5) * nx / 3 + rng.uniform(-0.18, 0.18) * nx / 3,
                    ],
                    dtype=float,
                )

                radii = np.array(
                    [
                        (nz / 2) * rng.uniform(0.18, 0.34),
                        (ny / 5) * rng.uniform(0.33, 0.58),
                        (nx / 3) * rng.uniform(0.32, 0.58),
                    ],
                    dtype=float,
                )

                family = pid % 6
                morphology_scale = (
                    np.array([1.35, 0.78, 1.00]) if family == 0 else
                    np.array([1.00, 1.25, 0.80]) if family == 1 else
                    np.array([0.82, 1.00, 1.25]) if family == 2 else
                    np.array([0.75, 1.20, 1.00]) if family == 3 else
                    np.array([1.00, 0.92, 0.92]) if family == 4 else
                    np.array([1.12, 0.88, 1.06])
                )
                radii *= morphology_scale

                specs.append(
                    {
                        "particle_id": pid,
                        "electrode": electrode_name,
                        "center_zyx_vox": center,
                        "base_radii_zyx_vox": radii,
                        "superquadric_p": float(rng.uniform(1.45, 3.20)),
                        "roughness": float(rng.uniform(0.05, 0.16)),
                        "frequencies": rng.integers(1, 5, size=3),
                        "phases": rng.uniform(0.0, 2.0 * np.pi, size=3),
                        "rotation": random_rotation_matrix(rng),
                        "family": family,
                    }
                )
                pid += 1

    assert len(specs) == 30
    return specs


def build_irregular_electrode(
    shape_zyx: tuple[int, int, int],
    *,
    rng: np.random.Generator,
    electrode_name: str,
    n_particles: int = 30,
    target_fraction: float = 0.70,
    skin_voxels: int = 1,
):
    specs = build_particle_specs(
        shape_zyx,
        rng=rng,
        n_particles=n_particles,
        electrode_name=electrode_name,
    )

    z, y, x = np.indices(shape_zyx, dtype=np.float32)
    points = np.stack((z, y, x), axis=-1).reshape(-1, 3)

    minimum_distance = np.full(len(points), np.inf, dtype=np.float32)
    nearest_particle = np.full(len(points), -1, dtype=np.int16)

    for spec in specs:
        local = (points - spec["center_zyx_vox"]) @ spec["rotation"]
        normalized = local / spec["base_radii_zyx_vox"]

        p = spec["superquadric_p"]
        distance = (
            np.abs(normalized[:, 0]) ** p
            + np.abs(normalized[:, 1]) ** p
            + np.abs(normalized[:, 2]) ** p
        ) ** (1.0 / p)

        radius = np.linalg.norm(normalized, axis=1) + 1.0e-7
        direction = normalized / radius[:, None]
        f0, f1, f2 = spec["frequencies"]
        ph0, ph1, ph2 = spec["phases"]
        modulation = (
            np.sin(f0 * np.arctan2(direction[:, 1], direction[:, 2]) + ph0)
            + np.cos(f1 * np.arctan2(direction[:, 0], direction[:, 2]) + ph1)
            + np.sin(f2 * np.pi * direction[:, 0] + ph2)
        ) / 3.0

        distance /= 1.0 + spec["roughness"] * modulation

        update = distance < minimum_distance
        minimum_distance[update] = distance[update]
        nearest_particle[update] = spec["particle_id"]

    minimum_distance = minimum_distance.reshape(shape_zyx)
    nearest_particle = nearest_particle.reshape(shape_zyx)

    eligible = np.ones(shape_zyx, dtype=bool)
    if skin_voxels:
        s = skin_voxels
        eligible[:s, :, :] = False
        eligible[-s:, :, :] = False
        eligible[:, :s, :] = False
        eligible[:, -s:, :] = False
        eligible[:, :, :s] = False
        eligible[:, :, -s:] = False

    target_count = int(round(target_fraction * np.prod(shape_zyx)))
    eligible_flat = np.flatnonzero(eligible)
    if target_count > len(eligible_flat):
        raise RuntimeError("active fraction is incompatible with the electrolyte skin")

    eligible_distance = minimum_distance.ravel()[eligible_flat]
    chosen_local = np.argpartition(eligible_distance, target_count - 1)[:target_count]
    chosen_flat = eligible_flat[chosen_local]

    active = np.zeros(minimum_distance.size, dtype=bool)
    active[chosen_flat] = True
    active = active.reshape(shape_zyx)

    contribution = np.bincount(
        nearest_particle[active].ravel(),
        minlength=n_particles,
    )
    if np.any(contribution == 0):
        raise RuntimeError(
            f"{electrode_name}: vanished particle IDs "
            f"{np.flatnonzero(contribution == 0).tolist()}"
        )

    contact_pairs: set[tuple[int, int]] = set()
    for axis in range(3):
        a_slice = [slice(None)] * 3
        b_slice = [slice(None)] * 3
        a_slice[axis] = slice(None, -1)
        b_slice[axis] = slice(1, None)

        owner_a = nearest_particle[tuple(a_slice)]
        owner_b = nearest_particle[tuple(b_slice)]
        contact = (
            active[tuple(a_slice)]
            & active[tuple(b_slice)]
            & (owner_a != owner_b)
        )
        for a, b in zip(owner_a[contact], owner_b[contact], strict=True):
            contact_pairs.add(tuple(sorted((int(a), int(b)))))

    degree = np.zeros(n_particles, dtype=int)
    for a, b in contact_pairs:
        degree[a] += 1
        degree[b] += 1

    _, active_components = ndi_label(
        active,
        structure=generate_binary_structure(3, 3),
    )

    table = pd.DataFrame(
        [
            {
                "electrode": electrode_name,
                "particle_id": spec["particle_id"],
                "family": spec["family"],
                "radius_z_um": spec["base_radii_zyx_vox"][0] * VOXEL_SIZE_M * 1e6,
                "radius_y_um": spec["base_radii_zyx_vox"][1] * VOXEL_SIZE_M * 1e6,
                "radius_x_um": spec["base_radii_zyx_vox"][2] * VOXEL_SIZE_M * 1e6,
                "superquadric_p": spec["superquadric_p"],
                "roughness": spec["roughness"],
                "owned_active_voxels": int(contribution[spec["particle_id"]]),
                "contact_degree": int(degree[spec["particle_id"]]),
            }
            for spec in specs
        ]
    )

    return {
        "active": active,
        "owner": nearest_particle,
        "specs": specs,
        "table": table,
        "contact_pairs": tuple(sorted(contact_pairs)),
        "active_components": int(active_components),
        "active_fraction": float(np.mean(active)),
    }


## 3. 生成严格三层体素结构

In [ ]:

rng = np.random.default_rng(SEED)

cathode = build_irregular_electrode(
    (NZ, NY, CATHODE_NX),
    rng=rng,
    electrode_name="cathode",
    n_particles=N_CATHODE_PARTICLES,
    target_fraction=TARGET_ACTIVE_FRACTION,
    skin_voxels=ELECTROLYTE_SKIN_VOXELS,
)
anode = build_irregular_electrode(
    (NZ, NY, ANODE_NX),
    rng=rng,
    electrode_name="anode",
    n_particles=N_ANODE_PARTICLES,
    target_fraction=TARGET_ACTIVE_FRACTION,
    skin_voxels=ELECTROLYTE_SKIN_VOXELS,
)

separator_start = CATHODE_NX
separator_stop = CATHODE_NX + SEPARATOR_NX

material_labels = np.full(
    (NZ, NY, NX),
    int(P.SEPARATOR_ELECTROLYTE),
    dtype=np.int32,
)

material_labels[:, :, :CATHODE_NX] = np.where(
    cathode["active"],
    int(P.POSITIVE_ACTIVE),
    int(P.POSITIVE_ELECTROLYTE),
)
material_labels[:, :, separator_start:separator_stop] = int(P.SEPARATOR_ELECTROLYTE)
material_labels[:, :, separator_stop:] = np.where(
    anode["active"],
    int(P.NEGATIVE_ACTIVE),
    int(P.NEGATIVE_ELECTROLYTE),
)

cathode_fraction = float(
    np.mean(material_labels[:, :, :CATHODE_NX] == int(P.POSITIVE_ACTIVE))
)
anode_fraction = float(
    np.mean(material_labels[:, :, separator_stop:] == int(P.NEGATIVE_ACTIVE))
)

assert abs(cathode_fraction - TARGET_ACTIVE_FRACTION) < 2.0 / cathode["active"].size
assert abs(anode_fraction - TARGET_ACTIVE_FRACTION) < 2.0 / anode["active"].size
assert len(cathode["table"]) == 30
assert len(anode["table"]) == 30

print(f"Cathode active fraction: {cathode_fraction:.8f}")
print(f"Anode active fraction:   {anode_fraction:.8f}")
print("Cathode particle contact pairs:", len(cathode["contact_pairs"]))
print("Anode particle contact pairs:  ", len(anode["contact_pairs"]))



## 4. 关键验证：60 个 tracking ID → 2 个活性材料域

为了证明“正极所有颗粒是一个区域、负极所有颗粒也是一个区域”不是 notebook 假象：

- 正极 30 个颗粒先分别标记为 `1000..1029`；
- 负极 30 个颗粒先分别标记为 `2000..2029`；
- 电解液和隔膜保留自己的材料 ID；
- `mesh_unstructured_regions(..., material_region_map=...)` 在 PLC 提取之前把：
  - `1000..1029 -> POSITIVE_ACTIVE`
  - `2000..2029 -> NEGATIVE_ACTIVE`

因此即使两个正极颗粒发生接触，接触处也不会被误认为“两个材料域”的界面。


In [ ]:

tracking_labels = material_labels.copy()

cathode_global = tracking_labels[:, :, :CATHODE_NX]
cathode_global[cathode["active"]] = (
    1000 + cathode["owner"][cathode["active"]]
).astype(np.int32)

anode_global = tracking_labels[:, :, separator_stop:]
anode_global[anode["active"]] = (
    2000 + anode["owner"][anode["active"]]
).astype(np.int32)

tracking_phase_names = {
    int(P.SEPARATOR_ELECTROLYTE): "separator_electrolyte",
    int(P.POSITIVE_ELECTROLYTE): "positive_electrolyte",
    int(P.NEGATIVE_ELECTROLYTE): "negative_electrolyte",
}
tracking_phase_names.update({1000 + i: f"positive_particle_{i:02d}" for i in range(30)})
tracking_phase_names.update({2000 + i: f"negative_particle_{i:02d}" for i in range(30)})

tracking_volume = MicrostructureVolume(
    labels=tracking_labels,
    voxel_size_m=(VOXEL_SIZE_M, VOXEL_SIZE_M, VOXEL_SIZE_M),
    origin_m=(0.0, 0.0, 0.0),
    phase_names=tracking_phase_names,
    metadata={
        "description": "60 tracked particles before FEM material collapse",
        "particle_count_cathode": 30,
        "particle_count_anode": 30,
    },
)

material_region_map = {
    int(P.SEPARATOR_ELECTROLYTE): int(P.SEPARATOR_ELECTROLYTE),
    int(P.POSITIVE_ELECTROLYTE): int(P.POSITIVE_ELECTROLYTE),
    int(P.NEGATIVE_ELECTROLYTE): int(P.NEGATIVE_ELECTROLYTE),
}
material_region_map.update({1000 + i: int(P.POSITIVE_ACTIVE) for i in range(30)})
material_region_map.update({2000 + i: int(P.NEGATIVE_ACTIVE) for i in range(30)})

material_volume = tracking_volume.remap_regions(
    material_region_map,
    region_names=PHASE_NAMES,
    require_complete=True,
)

print("Tracking phase count:", len(tracking_volume.phases))
print("FEM material phases :", material_volume.phases)

assert set(material_volume.phases) == {
    int(P.SEPARATOR_ELECTROLYTE),
    int(P.POSITIVE_ACTIVE),
    int(P.POSITIVE_ELECTROLYTE),
    int(P.NEGATIVE_ACTIVE),
    int(P.NEGATIVE_ELECTROLYTE),
}


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].imshow(
    material_labels[NZ // 2],
    origin="lower",
    aspect="auto",
    cmap="tab10",
)
axes[0, 0].set_title("Material phases: cathode | separator | anode")

axes[0, 1].imshow(
    tracking_labels[NZ // 2],
    origin="lower",
    aspect="auto",
    cmap="turbo",
)
axes[0, 1].set_title("60 particle tracking IDs before material collapse")

axes[1, 0].imshow(
    np.where(cathode["active"], cathode["owner"], np.nan)[NZ // 2],
    origin="lower",
    aspect="auto",
    cmap="tab20",
)
axes[1, 0].set_title("Cathode: 30 individual particles")

axes[1, 1].imshow(
    np.where(anode["active"], anode["owner"], np.nan)[NZ // 2],
    origin="lower",
    aspect="auto",
    cmap="tab20",
)
axes[1, 1].set_title("Anode: 30 individual particles")

plt.tight_layout()
plt.show()


## 5. 保存原始 tracking 体素与最终 FEM 材料体素

In [ ]:

tracking_outputs = tracking_volume.export(
    OUTPUT_DIR / "voxel_tracking",
    formats=("npz", "npy"),
)
material_outputs = material_volume.export(
    OUTPUT_DIR / "voxel_material",
    formats=("npz", "npy"),
)

print("Tracking outputs:", tracking_outputs)
print("Material outputs:", material_outputs)



## 6. 源码级 PLC 预检

最终 meshing 会自动做这一步。这里单独执行一次只是为了看到：

- diagonal voxel junction 数量；
- 最小 topology repair；
- `lock_plc_interfaces()` 是否真正锁定两个 separator 界面；
- 平滑后 PLC 是否仍然闭合、共形、无非流形边。


In [ ]:

ambiguous_before = count_nonmanifold_voxel_edges(material_volume.labels)
print("Ambiguous voxel edges before repair:", ambiguous_before)

preview_volume, preview_report = regularize_nonmanifold_junctions(
    material_volume,
    maximum_changed_fraction=0.01,
    maximum_iterations=30_000,
    preserve_outer_layer=True,
    minimum_phase_voxels=16,
    phase_change_penalties={
        int(P.SEPARATOR_ELECTROLYTE): 1.0e12,
    },
    strict=True,
)
print(preview_report)

assert preview_report.ambiguous_edges_after == 0
assert np.all(
    preview_volume.labels[:, :, separator_start:separator_stop]
    == int(P.SEPARATOR_ELECTROLYTE)
)

plc = extract_multiphase_plc(
    preview_volume,
    checkerboard_diagonals=True,
    preserve_outer_boundary=True,
    preserve_multiphase_junctions=True,
    strict=True,
)

locked_pairs = (
    (int(P.POSITIVE_ELECTROLYTE), int(P.SEPARATOR_ELECTROLYTE)),
    (int(P.SEPARATOR_ELECTROLYTE), int(P.NEGATIVE_ELECTROLYTE)),
)

plc_locked = lock_plc_interfaces(plc, locked_pairs)
plc_smoothed = smooth_multiphase_plc(
    plc_locked,
    iterations=6,
    relaxation=0.28,
    taubin_mu=-0.30,
    maximum_displacement_m=0.28 * VOXEL_SIZE_M,
)

plc_audit = audit_multiphase_plc(plc_smoothed)
print(plc_audit)
print("PLC vertices :", len(plc_smoothed.vertices))
print("PLC triangles:", len(plc_smoothed.triangles))
print("Locked interface pairs:", plc_smoothed.metadata["locked_interface_pairs"])

assert plc_audit.valid


## 7. 检查 TetGen C++ 原生扩展

In [ ]:

native_status = tetgen_native_status()
print(
    json.dumps(
        {
            "available": native_status.available,
            "version": native_status.version,
            "reason": native_status.reason,
            "module_path": (
                None
                if native_status.module_path is None
                else str(native_status.module_path)
            ),
            "license": native_status.license,
        },
        indent=2,
        ensure_ascii=False,
    )
)

if not native_status.available:
    print(json.dumps(
        tetgen_native_diagnostics(),
        indent=2,
        ensure_ascii=False,
        default=str,
    ))

if RUN_TETGEN and not native_status.available:
    raise RuntimeError(
        "TetGen native unavailable. Run scripts/diagnose_tetgen_native.py and reinstall "
        'with python -m pip install -e ".[zynmorph-tetgen]" -v.'
    )


## 8. 通用非结构化 TetGen 策略

这里不再直接用“最大四面体体积”硬编码所有区域，而是使用源码新增的 `IrregularMeshPolicy`：

- `base_edge_length_m` 控制全局体网格尺度；
- `region_edge_lengths_m` 控制不同材料内部目标尺度；
- `interface_edge_lengths_m` 控制复杂材料界面表面三角形尺度；
- `local_refinement_zones` 在指定空间位置进一步加密；
- `fixed_interface_pairs` 锁死必须保持精确几何的材料界面；
- 最终由 TetGen C++ 生成 constrained/conforming Delaunay 非结构化 Tet4。

同一 API 也可以直接接受任意 `MicrostructureVolume`、`MultiphasePLC` 或闭合 `SurfaceShell` 集合。

In [ ]:
um3 = 1.0e-18

center_y_m = 0.5 * NY * VOXEL_SIZE_M
center_z_m = 0.5 * NZ * VOXEL_SIZE_M

mesh_policy = IrregularMeshPolicy(
    # 全局 bulk 网格并不绑定 voxel cube；TetGen 可以自由生成 Delaunay Tet4。
    base_edge_length_m=1.80e-6,

    # 每个最终材料域只有一个 region ID，但可以有任意多个不连通分量。
    region_edge_lengths_m={
        int(P.SEPARATOR_ELECTROLYTE): 1.75e-6,
        int(P.POSITIVE_ACTIVE): 1.10e-6,
        int(P.POSITIVE_ELECTROLYTE): 0.85e-6,
        int(P.NEGATIVE_ACTIVE): 1.10e-6,
        int(P.NEGATIVE_ELECTROLYTE): 0.85e-6,
    },

    # 内部复杂界面比 bulk 更细。
    interface_edge_lengths_m={
        (int(P.POSITIVE_ACTIVE), int(P.POSITIVE_ELECTROLYTE)): 0.62e-6,
        (int(P.NEGATIVE_ACTIVE), int(P.NEGATIVE_ELECTROLYTE)): 0.62e-6,
        (int(P.POSITIVE_ELECTROLYTE), int(P.SEPARATOR_ELECTROLYTE)): 0.75e-6,
        (int(P.SEPARATOR_ELECTROLYTE), int(P.NEGATIVE_ELECTROLYTE)): 0.75e-6,
    },

    local_refinement_zones=(
        LocalRefinementZone(
            center_m_xyz=(
                (CATHODE_NX - 1.5) * VOXEL_SIZE_M,
                center_y_m,
                center_z_m,
            ),
            radius_m=6.0e-6,
            maximum_tetra_volume_m3=0.30 * um3,
            name="cathode_separator_refinement",
        ),
        LocalRefinementZone(
            center_m_xyz=(
                (CATHODE_NX + SEPARATOR_NX + 1.5) * VOXEL_SIZE_M,
                center_y_m,
                center_z_m,
            ),
            radius_m=6.0e-6,
            maximum_tetra_volume_m3=0.30 * um3,
            name="anode_separator_refinement",
        ),
    ),

    # 正极/隔膜与隔膜/负极宏观平面严格不动。
    fixed_interface_pairs=locked_pairs,

    radius_edge_ratio=1.45,
    minimum_dihedral_degrees=8.0,
    optimization_level=2,
    maximum_steiner_points=-1,

    regularize_junctions=True,
    junction_maximum_changed_fraction=0.01,
    junction_maximum_iterations=30_000,
    junction_minimum_phase_voxels=16,
    junction_preserve_outer_layer=True,
    junction_phase_change_penalties={
        int(P.SEPARATOR_ELECTROLYTE): 1.0e12,
    },

    smoothing_iterations=6,
    smoothing_relaxation=0.28,
    smoothing_taubin_mu=-0.30,
    maximum_surface_displacement_voxels=0.28,

    normalize_coordinates=True,
    consistency_check=True,
    conforming_delaunay=True,
    quiet=False,
)

tetgen_config = mesh_policy.to_tetgen_config(tracking_volume)
print(mesh_policy)
print("\nResolved TetGen config:")
print(tetgen_config)

## 9. 真正生成非规则、多域、自适应 Tet4

使用源码级统一入口 `mesh_complex_regions()`。这里给它的是带 60 个 tracking particle ID 的体素场，但通过 `material_region_map` 在 PLC 提取**之前**完成材料折叠：

- 30 个正极颗粒 tracking ID → 一个 `positive_active` material region；
- 30 个负极颗粒 tracking ID → 一个 `negative_active` material region；
- 不连通颗粒仍可使用多个 TetGen region seed，但所有 seed 携带相同材料 attribute；
- COMSOL 最终只看到一个正极活性材料 3D domain entity 和一个负极活性材料 3D domain entity。

In [ ]:

fem = None

if RUN_TETGEN:
    fem = mesh_complex_regions(
        tracking_volume,
        policy=mesh_policy,
        material_region_map=material_region_map,
        material_region_names=PHASE_NAMES,
        require_complete_region_map=True,
        maximum_tetrahedra=MAXIMUM_TETRAHEDRA,
    )

    print("Backend:", fem.backend)
    print("Nodes  :", fem.mesh.n_nodes)
    print("Tet4   :", fem.mesh.n_cells)
    print("Regions:", sorted(map(int, np.unique(fem.mesh.cell_regions))))
    print("Quality:", fem.quality)

    assert fem.backend.startswith("tetgen-1.6.0")
    assert fem.quality.fem_ready
    assert fem.quality.inverted_cells == 0
    assert fem.quality.degenerate_cells == 0

    # 最终 FEM 只有五个材料 ID；30+30 个颗粒 tracking IDs 已经消失。
    final_regions = set(map(int, np.unique(fem.mesh.cell_regions)))
    assert final_regions == set(PHASE_NAMES)
    assert not any(region >= 1000 for region in final_regions)

    # disconnected particles/components can have multiple TetGen seeds,
    # but every seed still carries the SAME material attribute.
    region_seeds = fem.metadata["region_seeds"]
    positive_seeds = [s for s in region_seeds if s.phase == int(P.POSITIVE_ACTIVE)]
    negative_seeds = [s for s in region_seeds if s.phase == int(P.NEGATIVE_ACTIVE)]

    print("Positive-active connected-component seeds:", len(positive_seeds))
    print("Negative-active connected-component seeds:", len(negative_seeds))
    assert all(s.phase == int(P.POSITIVE_ACTIVE) for s in positive_seeds)
    assert all(s.phase == int(P.NEGATIVE_ACTIVE) for s in negative_seeds)

    # 源码自动 topology repair 后，separator 仍必须是严格的完整长方体层。
    actual_meshed_volume = fem.metadata["meshed_volume"]
    assert np.all(
        actual_meshed_volume.labels[:, :, separator_start:separator_stop]
        == int(P.SEPARATOR_ELECTROLYTE)
    )
else:
    print(
        "TetGen execution skipped because ZYNNOVA_RUN_NATIVE_TETGEN=0. "
        "All generation, remap, topology, PLC and interface-lock gates were executed."
    )


## 10. 验证不是规则立方体拆分 Tet4

In [ ]:

if fem is not None:
    tet_volume = np.abs(tetrahedron_signed_volumes(fem.mesh))
    mean_ratio = tetrahedron_mean_ratio(fem.mesh)

    q = np.quantile(tet_volume, [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    cv = float(np.std(tet_volume) / np.mean(tet_volume))

    print("Tet volume quantiles [um^3]:", q * 1e18)
    print("q95/q05:", float(q[5] / q[1]))
    print("CV(volume):", cv)
    print("median mean-ratio:", float(np.median(mean_ratio)))

    assert q[5] / q[1] > 1.20
    assert cv > 0.05

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(tet_volume * 1e18, bins=80)
    axes[0].set_xlabel("Tet volume [um^3]")
    axes[0].set_ylabel("count")
    axes[0].set_title("Nonuniform TetGen element volumes")

    axes[1].hist(mean_ratio, bins=80)
    axes[1].set_xlabel("mean-ratio quality")
    axes[1].set_ylabel("count")
    axes[1].set_title("Tet4 quality")

    plt.tight_layout()
    plt.show()

    region_rows = []
    for region, name in PHASE_NAMES.items():
        values = tet_volume[fem.mesh.cell_regions == region]
        region_rows.append(
            {
                "region": region,
                "name": name,
                "tetrahedra": len(values),
                "q05_volume_um3": np.quantile(values, 0.05) * 1e18,
                "median_volume_um3": np.median(values) * 1e18,
                "q95_volume_um3": np.quantile(values, 0.95) * 1e18,
            }
        )
    display(pd.DataFrame(region_rows))



## 11. COMSOL MPHTXT：验证旧参数兼容 + 两极颗粒各为一个域

这里**故意继续使用**之前报错的：

```python
"include_boundary_triangles": True
```

新版源码会自动映射为 canonical `include_boundaries=True`，因此这个 cell 本身也是回归测试。


In [ ]:

exported = None
mphtxt_path = None

if fem is not None:
    domain_selections = {
        "cathode": (
            int(P.POSITIVE_ACTIVE),
            int(P.POSITIVE_ELECTROLYTE),
        ),
        "separator": (int(P.SEPARATOR_ELECTROLYTE),),
        "anode": (
            int(P.NEGATIVE_ACTIVE),
            int(P.NEGATIVE_ELECTROLYTE),
        ),
        "all_positive_particles": (int(P.POSITIVE_ACTIVE),),
        "all_negative_particles": (int(P.NEGATIVE_ACTIVE),),
    }

    exported = export_fem_mesh(
        fem,
        OUTPUT_DIR / "fem",
        formats=("mphtxt", "vtk", "msh", "inp"),
        export_boundary=True,
        comsol_domain_selections=domain_selections,
        comsol_options={
            # 历史参数：新版源码会兼容映射，不再 TypeError。
            "include_boundary_triangles": True,
            "include_interface_triangles": True,
            "include_domain_entity_indices": True,
            "verify": True,
        },
    )

    mphtxt_path = exported.exports["mphtxt"]
    info = inspect_comsol_mphtxt(mphtxt_path)

    print("Exports:")
    for key, path in exported.exports.items():
        print(f"  {key}: {path}")

    print("\nMPHTXT element counts:", info.element_counts)
    print("Tet domain entity IDs:", info.geometric_entity_ids["tet"])

    selections = {
        (selection.label, selection.dimension): selection.entity_ids
        for selection in info.selections
    }

    positive_entities = selections[("positive_active", 3)]
    negative_entities = selections[("negative_active", 3)]

    print("positive_active COMSOL entity IDs:", positive_entities)
    print("negative_active COMSOL entity IDs:", negative_entities)

    # 核心要求：30 个正极颗粒 = 一个 COMSOL 3D 材料实体；
    #          30 个负极颗粒 = 一个 COMSOL 3D 材料实体。
    assert len(positive_entities) == 1
    assert len(negative_entities) == 1
    assert positive_entities != negative_entities

    assert len(selections[("all_positive_particles", 3)]) == 1
    assert len(selections[("all_negative_particles", 3)]) == 1

    roundtrip = load_comsol_tet4_mphtxt(mphtxt_path)
    assert roundtrip.n_nodes == fem.mesh.n_nodes
    assert roundtrip.n_cells == fem.mesh.n_cells
    assert set(map(int, np.unique(roundtrip.cell_regions))) == set(PHASE_NAMES)

    print("\nCOMSOL round-trip passed.")


## 12. 最终审计报告

In [ ]:

summary = {
    "schema": "zynnova.complex-full-cell-unstructured.v2",
    "voxel_shape_zyx": [NZ, NY, NX],
    "voxel_size_m": VOXEL_SIZE_M,
    "particles": {
        "cathode_tracking_particles": 30,
        "anode_tracking_particles": 30,
        "cathode_active_fraction": cathode_fraction,
        "anode_active_fraction": anode_fraction,
        "cathode_contact_pairs": len(cathode["contact_pairs"]),
        "anode_contact_pairs": len(anode["contact_pairs"]),
    },
    "material_domain_collapse": {
        "tracking_phase_count": len(tracking_volume.phases),
        "final_material_phases": list(material_volume.phases),
        "positive_particle_tracking_ids": [1000 + i for i in range(30)],
        "negative_particle_tracking_ids": [2000 + i for i in range(30)],
        "positive_material_region": int(P.POSITIVE_ACTIVE),
        "negative_material_region": int(P.NEGATIVE_ACTIVE),
    },
    "plc_preflight": {
        "ambiguous_edges_before": ambiguous_before,
        "ambiguous_edges_after": preview_report.ambiguous_edges_after,
        "changed_fraction": preview_report.changed_fraction,
        "vertices": len(plc_smoothed.vertices),
        "triangles": len(plc_smoothed.triangles),
        "valid": plc_audit.valid,
        "locked_interface_pairs": plc_smoothed.metadata["locked_interface_pairs"],
    },
    "tetgen": {
        "requested": RUN_TETGEN,
        "native_available": native_status.available,
        "native_version": native_status.version,
    },
}

if fem is not None:
    tet_volume = np.abs(tetrahedron_signed_volumes(fem.mesh))
    summary["mesh"] = {
        "backend": fem.backend,
        "nodes": fem.mesh.n_nodes,
        "tetrahedra": fem.mesh.n_cells,
        "regions": sorted(map(int, np.unique(fem.mesh.cell_regions))),
        "inverted_cells": fem.quality.inverted_cells,
        "degenerate_cells": fem.quality.degenerate_cells,
        "median_mean_ratio": fem.quality.median_mean_ratio,
        "tet_volume_q05_um3": float(np.quantile(tet_volume, 0.05) * 1e18),
        "tet_volume_median_um3": float(np.median(tet_volume) * 1e18),
        "tet_volume_q95_um3": float(np.quantile(tet_volume, 0.95) * 1e18),
        "mphtxt": str(mphtxt_path),
    }

report_path = OUTPUT_DIR / "validation_summary.json"
report_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False, default=str))
print("\nValidation report:", report_path)
if mphtxt_path is not None:
    print("FINAL COMSOL MPHTXT:", mphtxt_path)
